# Medical Abstracts TC Corpus — model experiments

This notebook compares controlled variations of the baseline text classifier. The first experiment changes only class weighting.

Validation macro F1 is the primary selection metric. Accuracy and per-class precision, recall, and F1 help assess tradeoffs. TF-IDF is fitted exclusively on training data; the test set remains reserved for final evaluation.

Results apply to the filtered single-label subset created in notebook 02.

## 1. Environment setup

A fixed random state supports reproducibility. Run this notebook from the project root or the notebooks directory.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.pipeline import Pipeline

RANDOM_STATE = 42
TEXT_COL = "medical_abstract"
TARGET_COL = "condition_label"

CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == "notebooks" else CURRENT_DIR
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

## 2. Loading and validating the datasets

Load only training, validation, and label mapping files. Check missing values, empty texts, duplicates, class coverage, and overlap before training.

In [2]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
validation_df = pd.read_csv(PROCESSED_DATA_DIR / "validation.csv")
labels_df = pd.read_csv(PROCESSED_DATA_DIR / "labels.csv")

required_columns = {TEXT_COL, TARGET_COL}
expected_labels = set(labels_df[TARGET_COL])

for name, dataset in {"train": train_df, "validation": validation_df}.items():
    assert required_columns.issubset(dataset.columns)
    assert dataset[[TEXT_COL, TARGET_COL]].notna().all().all()
    assert dataset[TEXT_COL].map(lambda text: isinstance(text, str)).all()
    assert dataset[TEXT_COL].str.strip().ne("").all()
    assert not dataset[TEXT_COL].duplicated().any()
    assert set(dataset[TARGET_COL]) == expected_labels
    print(f"{name}: {dataset.shape}")

assert set(train_df[TEXT_COL]).isdisjoint(validation_df[TEXT_COL])

X_train = train_df[TEXT_COL]
y_train = train_df[TARGET_COL]
X_validation = validation_df[TEXT_COL]
y_validation = validation_df[TARGET_COL]

label_mapping = labels_df.set_index(TARGET_COL)["condition_name"].to_dict()
label_order = sorted(label_mapping)
target_names = [label_mapping[label] for label in label_order]

train: (5808, 2)
validation: (1245, 2)


## 3. Reproducing the baseline

Retrain the same TF-IDF and Logistic Regression configuration from notebook 03 so both candidates are compared in the same environment. No validation text is used to fit the vocabulary.

In [3]:
baseline_model = Pipeline(
    steps=[
        ("tfidf", TfidfVectorizer()),
        ("classifier", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ]
)

baseline_model.fit(X_train, y_train)
baseline_predictions = baseline_model.predict(X_validation)

## 4. Experiment: balanced class weights

Hypothesis: increasing the weight of errors on less frequent classes may improve their recall and validation macro F1.

The balanced setting computes weights inversely proportional to class frequencies in the training data. It changes the training objective, not the number of records. Higher recall may come at the cost of lower precision; improvement must be measured.

In [4]:
balanced_model = Pipeline(
    steps=[
        ("tfidf", TfidfVectorizer()),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

balanced_model.fit(X_train, y_train)
balanced_predictions = balanced_model.predict(X_validation)

## 5. Overall validation comparison

Compare measured metrics without assuming that class weighting improves the model.

In [5]:
predictions_by_model = {
    "baseline": baseline_predictions,
    "balanced": balanced_predictions,
}

results = pd.DataFrame(
    [
        {
            "model": name,
            "accuracy": accuracy_score(y_validation, predictions),
            "macro_f1": f1_score(
                y_validation, predictions, average="macro", zero_division=0
            ),
        }
        for name, predictions in predictions_by_model.items()
    ]
).set_index("model")

display(results)
print("Balanced minus baseline:")
display(results.loc["balanced"] - results.loc["baseline"])

,accuracy,macro_f1
model,,
baseline,0.781526,0.755347
balanced,0.799197,0.798036


Balanced minus baseline:


accuracy    0.017671
macro_f1    0.042690
dtype: float64

## 6. Per-class tradeoffs

Inspect precision, recall, and F1 for each class. Pay particular attention to digestive system diseases and nervous system diseases, whose baseline recall was low.

In [6]:
reports = {
    name: pd.DataFrame(
        classification_report(
            y_validation,
            predictions,
            labels=label_order,
            target_names=target_names,
            output_dict=True,
            zero_division=0,
        )
    ).T.loc[target_names, ["precision", "recall", "f1-score", "support"]]
    for name, predictions in predictions_by_model.items()
}

display(pd.concat(reports, names=["model", "class"]))

class_metric_changes = (
    reports["balanced"][["precision", "recall", "f1-score"]]
    - reports["baseline"][["precision", "recall", "f1-score"]]
)
display(class_metric_changes)

precision    recall  f1-score  \
model    class                                                            
baseline neoplasms                         0.905063  0.866667  0.885449   
         digestive system diseases         0.924242  0.580952  0.713450   
         nervous system diseases           0.817204  0.484076  0.608000   
         cardiovascular diseases           0.845395  0.874150  0.859532   
         general pathological conditions   0.628755  0.816156  0.710303   
balanced neoplasms                         0.914754  0.845455  0.878740   
         digestive system diseases         0.812500  0.866667  0.838710   
         nervous system diseases           0.689024  0.719745  0.704050   
         cardiovascular diseases           0.844156  0.884354  0.863787   
         general pathological conditions   0.707865  0.701950  0.704895   

                                          support  
model    class                                     
baseline neoplasms                          330.0  
         digestive system diseases          105.0  
         nervous system diseases            157.0  
         cardiovascular diseases            294.0  
         general pathological conditions    359.0  
balanced neoplasms                          330.0  
         digestive system diseases          105.0  
         nervous system diseases            157.0  
         cardiovascular diseases            294.0  
         general pathological conditions    359.0

,precision,recall,f1-score
neoplasms,0.009691,-0.021212,-0.006709
digestive system diseases,-0.111742,0.285714,0.125259
nervous system diseases,-0.128180,0.235669,0.096050
cardiovascular diseases,-0.001239,0.010204,0.004256
general pathological conditions,0.079110,-0.114206,-0.005408


## 7. Interpretation exercise

After running the notebook, replace these questions with your findings:

1. Did validation macro F1 improve? By how much?
2. Did recall improve for classes 2 and 3?
3. Which classes lost precision or F1?
4. Does the evidence support keeping balanced weights as a candidate?

Do not infer confidence or the cause of errors from aggregate metrics alone.

## 8. Next experiments

After reviewing this comparison, investigate word bigrams and regularization strength with controlled changes. Record the configuration and validation metrics for every candidate.

Keep the test set reserved until model selection is complete. This notebook does not yet establish a final model.

## 9. Experiment: balanced weights with word bigrams

This experiment extends the balanced model's TF-IDF representation to include
both individual words and consecutive two-word expressions.

The hypothesis is that these expressions provide useful context for
distinguishing medical categories.

Class weighting and classifier parameters remain unchanged, allowing us to
compare the effect of adding bigrams against the balanced unigram model.

In [7]:
balanced_bigram_model = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                ngram_range=(1, 2),
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

balanced_bigram_model.fit(X_train, y_train)

balanced_bigram_predictions = balanced_bigram_model.predict(
    X_validation
)

In [8]:
predictions_by_model["balanced_bigrams"] = balanced_bigram_predictions

results = pd.DataFrame(
    [
        {
            "model": name,
            "accuracy": accuracy_score(y_validation, predictions),
            "macro_f1": f1_score(
                y_validation,
                predictions,
                average="macro",
                zero_division=0,
            ),
        }
        for name, predictions in predictions_by_model.items()
    ]
).set_index("model")

display(results)

print("Balanced bigrams minus balanced unigrams:")
display(
    results.loc["balanced_bigrams"]
    - results.loc["balanced"]
)

,accuracy,macro_f1
model,,
baseline,0.781526,0.755347
balanced,0.799197,0.798036
balanced_bigrams,0.806426,0.806356


Balanced bigrams minus balanced unigrams:


accuracy    0.007229
macro_f1    0.008320
dtype: float64

In [9]:
bigram_report = pd.DataFrame(
    classification_report(
        y_validation,
        balanced_bigram_predictions,
        labels=label_order,
        target_names=target_names,
        output_dict=True,
        zero_division=0,
    )
).T.loc[
    target_names,
    ["precision", "recall", "f1-score", "support"],
]

display(bigram_report)

print("Per-class changes compared with balanced unigrams:")
display(
    bigram_report[["precision", "recall", "f1-score"]]
    - reports["balanced"][["precision", "recall", "f1-score"]]
)

,precision,recall,f1-score,support
neoplasms,0.900312,0.875758,0.887865,330.0
digestive system diseases,0.845455,0.885714,0.865116,105.0
nervous system diseases,0.695122,0.726115,0.710280,157.0
cardiovascular diseases,0.824841,0.880952,0.851974,294.0
general pathological conditions,0.741071,0.693593,0.716547,359.0


Per-class changes compared with balanced unigrams:


,precision,recall,f1-score
neoplasms,-0.014443,0.030303,0.009125
digestive system diseases,0.032955,0.019048,0.026407
nervous system diseases,0.006098,0.006369,0.006231
cardiovascular diseases,-0.019315,-0.003401,-0.011814
general pathological conditions,0.033206,-0.008357,0.011652


In [10]:
for name, model in {
    "balanced_unigrams": balanced_model,
    "balanced_bigrams": balanced_bigram_model,
}.items():
    vocabulary_size = len(
        model.named_steps["tfidf"].vocabulary_
    )
    print(f"{name}: {vocabulary_size:,} features")

balanced_unigrams: 29,215 features
balanced_bigrams: 401,245 features


## 10. Experiment: filtering rare terms

This experiment keeps balanced class weights and word bigrams, but sets
`min_df=2`. Terms must appear in at least two training documents to be
included in the vocabulary.

The hypothesis is that filtering rare terms reduces the feature space
while preserving validation performance.

Only training documents determine which terms are retained.

In [11]:
balanced_bigram_min_df_model = Pipeline(
    steps=[
        (
            "tfidf",
            TfidfVectorizer(
                ngram_range=(1, 2),
                min_df=2,
            ),
        ),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=1000,
                random_state=RANDOM_STATE,
            ),
        ),
    ]
)

balanced_bigram_min_df_model.fit(X_train, y_train)

min_df_predictions = balanced_bigram_min_df_model.predict(
    X_validation
)

In [12]:
predictions_by_model["balanced_bigrams_min_df2"] = min_df_predictions

models = {
    "baseline": baseline_model,
    "balanced": balanced_model,
    "balanced_bigrams": balanced_bigram_model,
    "balanced_bigrams_min_df2": balanced_bigram_min_df_model,
}

results = pd.DataFrame(
    [
        {
            "model": name,
            "accuracy": accuracy_score(
                y_validation,
                predictions_by_model[name],
            ),
            "macro_f1": f1_score(
                y_validation,
                predictions_by_model[name],
                average="macro",
                zero_division=0,
            ),
            "features": len(model.named_steps["tfidf"].vocabulary_),
        }
        for name, model in models.items()
    ]
).set_index("model")

display(results)

,accuracy,macro_f1,features
model,,,
baseline,0.781526,0.755347,29215
balanced,0.799197,0.798036,29215
balanced_bigrams,0.806426,0.806356,401245
balanced_bigrams_min_df2,0.807229,0.807653,104360


In [13]:
min_df_report = pd.DataFrame(
    classification_report(
        y_validation,
        min_df_predictions,
        labels=label_order,
        target_names=target_names,
        output_dict=True,
        zero_division=0,
    )
).T.loc[
    target_names,
    ["precision", "recall", "f1-score", "support"],
]

display(min_df_report)

print("Changes compared with unfiltered bigrams:")
display(
    min_df_report[["precision", "recall", "f1-score"]]
    - bigram_report[["precision", "recall", "f1-score"]]
)

,precision,recall,f1-score,support
neoplasms,0.914286,0.872727,0.893023,330.0
digestive system diseases,0.846847,0.895238,0.870370,105.0
nervous system diseases,0.691358,0.713376,0.702194,157.0
cardiovascular diseases,0.843137,0.877551,0.860000,294.0
general pathological conditions,0.720798,0.704735,0.712676,359.0


Changes compared with unfiltered bigrams:


,precision,recall,f1-score
neoplasms,0.013974,-0.003030,0.005158
digestive system diseases,0.001392,0.009524,0.005254
nervous system diseases,-0.003764,-0.012739,-0.008086
cardiovascular diseases,0.018296,-0.003401,0.008026
general pathological conditions,-0.020274,0.011142,-0.003871


### Findings

Setting `min_df=2` reduced the vocabulary from 401,245 to 104,360
features, a reduction of approximately 74%.

Validation macro F1 increased slightly from 0.8064 to 0.8077.
Per-class F1 improved for neoplasms, digestive system diseases, and
cardiovascular diseases, while decreasing for nervous system diseases
and general pathological conditions.

The main benefit is the substantially smaller vocabulary with similar
validation performance. Inference latency has not yet been measured.

This configuration is the preferred candidate so far.

## 11. Experiment: regularization strength

This experiment compares Logistic Regression regularization strengths
using `C=0.1`, `C=1.0`, and `C=10.0`.

Lower values of C apply stronger regularization. TF-IDF settings and
balanced class weights remain unchanged.

Training and validation macro F1 are recorded to examine the
generalization gap. Model selection uses validation macro F1.

In [14]:
regularization_models = {}
regularization_predictions = {}
regularization_rows = []

for c_value in [0.1, 1.0, 10.0]:
    model = Pipeline(
        steps=[
            (
                "tfidf",
                TfidfVectorizer(
                    ngram_range=(1, 2),
                    min_df=2,
                ),
            ),
            (
                "classifier",
                LogisticRegression(
                    C=c_value,
                    class_weight="balanced",
                    max_iter=1000,
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )

    model.fit(X_train, y_train)

    train_predictions = model.predict(X_train)
    validation_predictions = model.predict(X_validation)

    train_macro_f1 = f1_score(
        y_train,
        train_predictions,
        average="macro",
        zero_division=0,
    )

    validation_macro_f1 = f1_score(
        y_validation,
        validation_predictions,
        average="macro",
        zero_division=0,
    )

    regularization_models[c_value] = model
    regularization_predictions[c_value] = validation_predictions

    regularization_rows.append(
        {
            "C": c_value,
            "train_macro_f1": train_macro_f1,
            "validation_macro_f1": validation_macro_f1,
            "validation_accuracy": accuracy_score(
                y_validation,
                validation_predictions,
            ),
            "f1_gap": train_macro_f1 - validation_macro_f1,
        }
    )

regularization_results = pd.DataFrame(
    regularization_rows
).set_index("C")

display(regularization_results)

,train_macro_f1,validation_macro_f1,validation_accuracy,f1_gap
C,,,,
0.1,0.848356,0.755281,0.762249,0.093075
1.0,0.953944,0.807653,0.807229,0.146291
10.0,1.000000,0.813545,0.817671,0.186455


In [16]:
best_c = regularization_results[
    "validation_macro_f1"
].idxmax()

best_regularization_model = regularization_models[best_c]

best_regularization_predictions = regularization_predictions[
    best_c
]

print(f"Best C on validation: {best_c}")

Best C on validation: 10.0


In [17]:
regularization_report = pd.DataFrame(
    classification_report(
        y_validation,
        best_regularization_predictions,
        labels=label_order,
        target_names=target_names,
        output_dict=True,
        zero_division=0,
    )
).T.loc[
    target_names,
    ["precision", "recall", "f1-score", "support"],
]

display(regularization_report)

print("Changes compared with the previous C=1 candidate:")
display(
    regularization_report[["precision", "recall", "f1-score"]]
    - min_df_report[["precision", "recall", "f1-score"]]
)

,precision,recall,f1-score,support
neoplasms,0.923077,0.872727,0.897196,330.0
digestive system diseases,0.862745,0.838095,0.850242,105.0
nervous system diseases,0.727891,0.681529,0.703947,157.0
cardiovascular diseases,0.870000,0.887755,0.878788,294.0
general pathological conditions,0.713542,0.763231,0.737550,359.0


Changes compared with the previous C=1 candidate:


,precision,recall,f1-score
neoplasms,0.008791,0.000000,0.004173
digestive system diseases,0.015898,-0.057143,-0.020129
nervous system diseases,0.036533,-0.031847,0.001753
cardiovascular diseases,0.026863,0.010204,0.018788
general pathological conditions,-0.007256,0.058496,0.024874


### Findings

C=10 achieved the highest validation macro F1 among the tested values,
reaching 0.8135, compared with 0.8077 for C=1.

Training macro F1 reached 1.0000, increasing the training–validation
gap. This warrants caution when interpreting generalization.

The improvement was not uniform across classes. Digestive system
diseases lost F1, while cardiovascular diseases and general pathological
conditions improved.

C=10 is the selected configuration based on validation macro F1.
The test set has not been used for model selection.